In [1]:
import os, sys, json, hashlib, shutil, subprocess, time, random
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import cv2
from PIL import Image
from tqdm.auto import tqdm

REPO_URL = "https://github.com/Mugdha-Naik/deepfake-detector-project.git"
REPO_DIR = "/kaggle/working/repo"
if not os.path.exists(REPO_DIR):
    os.system(f"git clone {REPO_URL} {REPO_DIR}")
else:
    os.system(f"cd {REPO_DIR} && git pull")

INPUT_ROOT = "/kaggle/input"

def find_one(name, roots):
    """Exactly one copy under the first root that has any, else an error."""
    for root in roots:
        matches = [os.path.join(d, name) for d, _, fs in os.walk(root) if name in fs]
        if len(matches) == 1:
            return matches[0]
        if len(matches) > 1:
            raise RuntimeError(f"{name}: {len(matches)} copies under {root}. Attach only one:\n" + "\n".join(matches))
    raise FileNotFoundError(f"{name} not found under {roots}")

def resolve_dir(name, root):
    """Find exactly one folder called `name` (does not descend into it)."""
    found = []
    for d, dirnames, _ in os.walk(root):
        if name in dirnames:
            found.append(os.path.join(d, name))
            dirnames.remove(name)
    if len(found) != 1:
        raise RuntimeError(f"expected exactly one '{name}' folder under {root}, found: {found}")
    return found[0]

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

print("helpers ready")

Cloning into '/kaggle/working/repo'...


helpers ready


In [2]:
# ============================================================
# LOAD THE FROZEN MANIFEST AND PROVE IT IS THE FROZEN ONE
# ============================================================
manifest_csv = find_one("faces_manifest.csv", [REPO_DIR, INPUT_ROOT])
meta_json = find_one("manifest_meta.json", [REPO_DIR, INPUT_ROOT])
meta = json.load(open(meta_json))
assert sha256(manifest_csv) == meta["manifest_sha256"], (
    "faces_manifest.csv does not match manifest_meta.json. It was modified (for example re-saved in Excel) "
    "or is a different version. Use the exact file produced by step 2.1.")
m = pd.read_csv(manifest_csv)
print("manifest:", manifest_csv, "| rows:", len(m), "| sha256 verified")

CLEAN_ROOT = resolve_dir("step5_faces", INPUT_ROOT)
print("clean crops:", CLEAN_ROOT)

clean = m[m["role"].isin(["test_A", "test_C"])].copy()
clean["clean_abs"] = clean["face_rel"].apply(lambda r: os.path.join(CLEAN_ROOT, r))
missing_clean = [p for p in clean["clean_abs"] if not os.path.exists(p)]
assert not missing_clean, f"{len(missing_clean)} clean crops missing, e.g. {missing_clean[:2]}"

videos = clean["video_path"].unique()
missing_vid = [v for v in videos if not os.path.exists(v)]
assert not missing_vid, f"attach the Celeb-DF and FaceForensics++ datasets; missing e.g. {missing_vid[:2]}"
print(clean.groupby(["role", "label_name"]).agg(videos=("video_id", "nunique"), faces=("face_id", "size")))

manifest: /kaggle/input/datasets/mugdhanaik/faces-manifest/faces_manifest.csv | rows: 28184 | sha256 verified
clean crops: /kaggle/input/notebooks/mugdhanaik/step5-regenerate/step5_faces
                   videos  faces
role   label_name               
test_A FAKE           100   3000
       REAL            98   2883
test_C FAKE           150   4382
       REAL           150   4381


In [3]:
# ============================================================
# ffmpeg + the ONE compression setting (record it, do not change it between runs)
# ============================================================
CRF, PRESET = 40, "veryfast"

def get_ffmpeg():
    exe = shutil.which("ffmpeg")
    if exe:
        return exe
    os.system("pip install -q imageio-ffmpeg")
    import imageio_ffmpeg
    return imageio_ffmpeg.get_ffmpeg_exe()

FFMPEG = get_ffmpeg()
FFMPEG_VERSION = subprocess.run([FFMPEG, "-version"], capture_output=True, text=True).stdout.splitlines()[0]
print(FFMPEG_VERSION)

def reencode(src, dst):
    """H.264, constant rate factor CRF, no audio, same frame count (passthrough), even dimensions."""
    base = [FFMPEG, "-y", "-v", "error", "-i", src, "-an",
            "-vf", "scale=trunc(iw/2)*2:trunc(ih/2)*2",
            "-c:v", "libx264", "-preset", PRESET, "-crf", str(CRF), "-pix_fmt", "yuv420p"]
    err = ""
    for sync in (["-fps_mode", "passthrough"], ["-vsync", "0"]):      # newer / older ffmpeg spelling
        r = subprocess.run(base + sync + [dst], capture_output=True, text=True)
        if r.returncode == 0:
            return True, ""
        err = r.stderr[-300:]
    return False, err

def video_info(path):
    cap = cv2.VideoCapture(path)
    info = (int(cap.get(cv2.CAP_PROP_FRAME_COUNT)), int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)))
    cap.release()
    return info

ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers


In [4]:
# ============================================================
# RE-ENCODE EACH TEST VIDEO, CROP THE SAME FRAMES AND BOXES, DELETE THE TEMP VIDEO
# (re-runnable: a video whose crops already exist is skipped)
# ============================================================
OUT = "/kaggle/working"
CROP_ROOT = os.path.join(OUT, "compressed_faces")
TMP = "/tmp/p2_2"
os.makedirs(TMP, exist_ok=True)
COND = {"test_A": ("B", "faces_B"), "test_C": ("D", "faces_D")}
for sub in ("faces_B", "faces_D"):
    os.makedirs(os.path.join(CROP_ROOT, sub), exist_ok=True)

clean["tag"] = clean["role"].map(lambda r: COND[r][0])
clean["new_face_id"] = clean["face_id"] + "__" + clean["tag"]
clean["new_rel"] = clean["role"].map(lambda r: COND[r][1]) + "/" + clean["new_face_id"] + ".jpg"
clean["new_abs"] = clean["new_rel"].apply(lambda r: os.path.join(CROP_ROOT, r))

problems, enc_rows = [], []
groups = list(clean.groupby("video_path", sort=False))
t0 = time.time()

for n, (vp, g) in enumerate(tqdm(groups, desc="videos"), 1):
    todo = g[~g["new_abs"].apply(os.path.exists)].sort_values("frame_number")
    if todo.empty:
        continue
    vid = g["video_id"].iloc[0]
    tmp = os.path.join(TMP, f"{vid}.mp4")
    try:
        t_v = time.time()
        ok, err = reencode(vp, tmp)
        if not ok:
            raise RuntimeError("ffmpeg failed: " + err)
        src_info, dst_info = video_info(vp), video_info(tmp)
        if src_info != dst_info:
            raise RuntimeError(f"video changed during encoding (frames, width, height): {src_info} -> {dst_info}")
        cap = cv2.VideoCapture(tmp)
        for r in todo.itertuples():
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(r.frame_number))
            ok, frame = cap.read()
            if not ok:
                raise RuntimeError(f"frame {r.frame_number} could not be read")
            face = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)[int(r.y1):int(r.y2), int(r.x1):int(r.x2)]
            if face.shape[0] != int(r.face_height) or face.shape[1] != int(r.face_width):
                raise RuntimeError(f"crop {face.shape[:2]} != ({r.face_height}, {r.face_width}) at frame {r.frame_number}")
            Image.fromarray(face).save(r.new_abs, quality=95)
        cap.release()
        enc_rows.append({"video_id": vid, "role": g["role"].iloc[0], "frames": src_info[0], "width": src_info[1],
                         "height": src_info[2], "src_mb": round(os.path.getsize(vp) / 1e6, 2),
                         "compressed_mb": round(os.path.getsize(tmp) / 1e6, 3), "seconds": round(time.time() - t_v, 1)})
    except Exception as e:
        problems.append((vid, vp, f"{type(e).__name__}: {e}"[:220]))
    finally:
        if os.path.exists(tmp):
            os.remove(tmp)
    if n % 25 == 0:
        eta = (time.time() - t0) / n * (len(groups) - n) / 60
        print(f"[{n}/{len(groups)}] problems so far: {len(problems)} | ETA ~{eta:.0f} min")

enc_df = pd.DataFrame(enc_rows)
print("Done in", round((time.time() - t0) / 60, 1), "min | videos encoded this run:", len(enc_df), "| problems:", len(problems))
if problems:
    pd.DataFrame(problems, columns=["video_id", "video_path", "problem"]).to_csv(os.path.join(OUT, "p2_2_problems.csv"), index=False)
    print(pd.DataFrame(problems, columns=["video_id", "video_path", "problem"]).head(10).to_string())
assert not problems, "some videos failed (see p2_2_problems.csv): a missing twin breaks the pairing, so fix and re-run"

videos:   0%|          | 0/498 [00:00<?, ?it/s]

[25/498] problems so far: 0 | ETA ~45 min
[50/498] problems so far: 0 | ETA ~48 min
[75/498] problems so far: 0 | ETA ~47 min
[100/498] problems so far: 0 | ETA ~47 min
[125/498] problems so far: 0 | ETA ~44 min
[150/498] problems so far: 0 | ETA ~41 min
[175/498] problems so far: 0 | ETA ~37 min
[200/498] problems so far: 0 | ETA ~34 min
[225/498] problems so far: 0 | ETA ~29 min
[250/498] problems so far: 0 | ETA ~25 min
[275/498] problems so far: 0 | ETA ~21 min
[300/498] problems so far: 0 | ETA ~18 min
[325/498] problems so far: 0 | ETA ~15 min
[350/498] problems so far: 0 | ETA ~12 min
[375/498] problems so far: 0 | ETA ~10 min
[400/498] problems so far: 0 | ETA ~8 min
[425/498] problems so far: 0 | ETA ~6 min
[450/498] problems so far: 0 | ETA ~4 min
[475/498] problems so far: 0 | ETA ~2 min
Done in 36.5 min | videos encoded this run: 498 | problems: 0


In [5]:
# ============================================================
# BUILD THE B AND D MANIFESTS AND RUN THE CHECKS
# ============================================================
def blur_of(path):
    img = cv2.imread(path)
    return float(cv2.Laplacian(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY), cv2.CV_64F).var())

clean["blur_lapvar"] = clean["new_abs"].apply(blur_of)

def build(role, tag):
    d = clean[clean["role"] == role]
    return pd.DataFrame({
        "face_id": d["new_face_id"].values, "pair_id": d["face_id"].values, "face_rel": d["new_rel"].values,
        "video_id": d["video_id"].values, "video_key": d["video_key"].values, "video_path": d["video_path"].values,
        "dataset": d["dataset"].values, "method": d["method"].values, "label_name": d["label_name"].values,
        "label": d["label"].values, "split": d["split"].values, "role": f"test_{tag}",
        "condition": f"{tag}_compressed", "frame_number": d["frame_number"].values,
        "x1": d["x1"].values, "y1": d["y1"].values, "x2": d["x2"].values, "y2": d["y2"].values,
        "face_width": d["face_width"].values, "face_height": d["face_height"].values,
        "blur_lapvar": d["blur_lapvar"].round(3).values, "crf": CRF, "preset": PRESET})

manifests = {"B": build("test_A", "B"), "D": build("test_C", "D")}

results = []
def check(name, ok, detail=""):
    results.append((name, bool(ok), detail))
    print(("PASS  " if ok else "FAIL  ") + name + (f"   [{detail}]" if detail else ""))

for tag, role, ds in (("B", "test_A", "FaceForensics++"), ("D", "test_C", "Celeb-DF")):
    mb, src = manifests[tag], m[m["role"] == role]
    check(f"{tag}: one compressed face for every clean {role} face", len(mb) == len(src) and set(mb["pair_id"]) == set(src["face_id"]),
          f"{len(mb)} vs {len(src)}")
    check(f"{tag}: face_id and pair_id are unique", mb["face_id"].is_unique and mb["pair_id"].is_unique)
    j = mb.merge(src, left_on="pair_id", right_on="face_id", suffixes=("", "_c"))
    same = all((j[c] == j[c + "_c"]).all() for c in ["video_id", "frame_number", "x1", "y1", "x2", "y2", "label", "split", "dataset"])
    check(f"{tag}: twins share video, frame, box, label, split, dataset", same)
    check(f"{tag}: only {ds} test faces", set(mb["dataset"]) == {ds} and set(mb["split"]) == {"test"})
    missing_new = [p for p in clean.loc[clean["role"] == role, "new_abs"] if not os.path.exists(p)]
    check(f"{tag}: every compressed crop exists", not missing_new, f"{len(missing_new)} missing")
    bad = 0
    pool = list(clean.loc[clean["role"] == role, "new_abs"])
    for p in random.Random(0).sample(pool, min(200, len(pool))):
        try:
            with Image.open(p) as im:
                im.verify()
        except Exception:
            bad += 1
    check(f"{tag}: random 200 compressed crops open as images", bad == 0, f"{bad} unreadable")

# did compression actually change the images? compare sharpness of twins with the SAME function
samp = clean.sample(min(600, len(clean)), random_state=0)
clean_b = np.array([blur_of(p) for p in samp["clean_abs"]])
new_b = samp["blur_lapvar"].to_numpy()
ratio = float(np.median(new_b / np.maximum(clean_b, 1e-6)))
check("compression lowered sharpness (median twin sharpness ratio < 1)", ratio < 1.0, f"median compressed/clean = {ratio:.3f}")
if len(enc_df):
    check("frame count, width, height unchanged for every encoded video", True, f"{len(enc_df)} videos verified during encoding")

failed = [n for n, ok, _ in results if not ok]
print("\n" + "=" * 60)
print(f"{len(results) - len(failed)} of {len(results)} checks passed")
assert not failed, f"CHECKS FAILED: {failed}"
print("ALL CHECKS PASSED")

PASS  B: one compressed face for every clean test_A face   [5883 vs 5883]
PASS  B: face_id and pair_id are unique
PASS  B: twins share video, frame, box, label, split, dataset
PASS  B: only FaceForensics++ test faces
PASS  B: every compressed crop exists   [0 missing]
PASS  B: random 200 compressed crops open as images   [0 unreadable]
PASS  D: one compressed face for every clean test_C face   [8763 vs 8763]
PASS  D: face_id and pair_id are unique
PASS  D: twins share video, frame, box, label, split, dataset
PASS  D: only Celeb-DF test faces
PASS  D: every compressed crop exists   [0 missing]
PASS  D: random 200 compressed crops open as images   [0 unreadable]
PASS  compression lowered sharpness (median twin sharpness ratio < 1)   [median compressed/clean = 0.939]
PASS  frame count, width, height unchanged for every encoded video   [498 videos verified during encoding]

14 of 14 checks passed
ALL CHECKS PASSED


In [6]:
# ============================================================
# SAVE manifests, metadata, encode stats and a visual side-by-side sample
# ============================================================
for tag, mb in manifests.items():
    mb.to_csv(os.path.join(OUT, f"faces_manifest_{tag}.csv"), index=False)
if len(enc_df):
    enc_df.to_csv(os.path.join(OUT, "encode_stats.csv"), index=False)

meta2 = {
    "created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "clean_manifest_sha256": meta["manifest_sha256"],
    "ffmpeg": FFMPEG_VERSION,
    "encode": {"codec": "libx264", "crf": CRF, "preset": PRESET, "pix_fmt": "yuv420p", "audio": "removed",
               "note": "re-encode of C23 videos: a double-compression approximation of C40, not the official C40"},
    "faces": {tag: int(len(mb)) for tag, mb in manifests.items()},
    "median_twin_sharpness_ratio": round(ratio, 4),
    "manifest_sha256": {tag: sha256(os.path.join(OUT, f"faces_manifest_{tag}.csv")) for tag in manifests},
    "face_rel_note": "face_rel is relative to the 'compressed_faces' folder; resolve it with resolve_dir('compressed_faces', ...)",
}
with open(os.path.join(OUT, "conditions_meta.json"), "w") as f:
    json.dump(meta2, f, indent=2)

pairs = pd.concat([manifests["B"].sample(4, random_state=1), manifests["D"].sample(4, random_state=1)])
canvas = Image.new("RGB", (8 * 160, 2 * 160), "white")
for i, r in enumerate(pairs.itertuples()):
    twin = clean[clean["new_face_id"] == r.face_id].iloc[0]
    canvas.paste(Image.open(twin["clean_abs"]).convert("RGB").resize((160, 160)), (i * 160, 0))
    canvas.paste(Image.open(os.path.join(CROP_ROOT, r.face_rel)).convert("RGB").resize((160, 160)), (i * 160, 160))
canvas.save(os.path.join(OUT, "pair_samples.png"))

print("Saved:")
for n in ("faces_manifest_B.csv", "faces_manifest_D.csv", "conditions_meta.json", "encode_stats.csv", "pair_samples.png"):
    print(" ", n)
print("compressed crops:", sum(len(fs) for _, _, fs in os.walk(CROP_ROOT)), "files in", CROP_ROOT)
print(json.dumps({k: meta2[k] for k in ("faces", "median_twin_sharpness_ratio", "manifest_sha256")}, indent=2))
if len(enc_df):
    print(enc_df.groupby("role")[["src_mb", "compressed_mb", "seconds"]].mean().round(2))

Saved:
  faces_manifest_B.csv
  faces_manifest_D.csv
  conditions_meta.json
  encode_stats.csv
  pair_samples.png
compressed crops: 14646 files in /kaggle/working/compressed_faces
{
  "faces": {
    "B": 5883,
    "D": 8763
  },
  "median_twin_sharpness_ratio": 0.9391,
  "manifest_sha256": {
    "B": "c4454a9f9997762dd667ca966bf425a4bc6fc948b36321005a38ff26626d2605",
    "D": "d192e38e54fb2d9d6c8a2a03ddb9ffe322512069190bb7178614798749c751f6"
  }
}
        src_mb  compressed_mb  seconds
role                                  
test_A    1.88           0.15     6.84
test_C    1.68           0.08     2.78
